# Notebook 04 — Eigenvalues and Eigenvectors

**Module:** 04 — Linear Algebra  
**Notebook:** 04 of 10  
**Prerequisites:** NB02, NB03  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

PCA reduces dimensionality by finding the directions of maximum variance in the data.
Those directions are the **eigenvectors** of the covariance matrix, and the variance
explained is the corresponding **eigenvalue**. Understanding eigenvectors geometrically
is the key to understanding PCA, SVD, spectral clustering, and PageRank.

---
## Step 2 — Intuition

When you multiply a vector $\mathbf{v}$ by a matrix $A$, it generally rotates and
scales. An **eigenvector** is special: it doesn't rotate — it only gets scaled.
The scaling factor is the **eigenvalue** $\lambda$:

$$A\mathbf{v} = \lambda \mathbf{v}$$

For the covariance matrix of gene expression data:
- The **first eigenvector** points in the direction that explains the most variance
  (the direction genes vary most across samples).
- The **eigenvalue** tells you how much variance that direction captures.
- Eigenvectors of the covariance matrix are always **orthogonal** (because the
  covariance matrix is symmetric and positive semi-definite).

---
## Step 3 — Biological Background

**PCA of gene expression:**
- The covariance matrix $\mathbf{C} \in \mathbb{R}^{S \times S}$ (samples × samples,
  for computational tractability) has eigenvectors called **principal components**.
- PC1 captures the largest source of variation — often batch effects, cell cycle,
  or true biological condition.
- PC2 captures the second-largest source, orthogonal to PC1.

**Spectral clustering of gene regulatory networks:**
- The graph Laplacian's eigenvectors (Fiedler vector) reveal community structure
  — which genes form functional modules. This is NB07.

**Population genetics:**
- PCA on a genotype matrix stratifies samples by ancestry. The first PC often
  separates populations (Novembre et al., 2008 — 'genes mirror geography').

---
## Step 4 — Mathematical Explanation

**Definition:** $\mathbf{v}$ is an eigenvector of $A$ with eigenvalue $\lambda$ if:
$$A\mathbf{v} = \lambda\mathbf{v}, \quad \mathbf{v} \neq \mathbf{0}$$

**Characteristic equation:** $\det(A - \lambda I) = 0$.
Solving this gives the eigenvalues; substituting back gives the eigenvectors.

**Spectral decomposition** (for symmetric $A$):
$$A = Q\Lambda Q^\top$$
where $Q$ is the matrix of eigenvectors (columns) and $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

**Properties for symmetric positive semi-definite matrices (covariance matrices):**
- All eigenvalues $\lambda_i \geq 0$
- All eigenvectors are real and mutually orthogonal
- $\sum_i \lambda_i = \text{trace}(A)$ (total variance)
- Fraction of variance explained by PC $i$: $\lambda_i / \sum_j \lambda_j$

**Power iteration:** the simplest algorithm for finding the dominant eigenvector:
start with a random vector, repeatedly multiply by $A$ and normalize. Converges to the
eigenvector with the largest eigenvalue.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Power iteration from scratch
def power_iteration(A: np.ndarray, n_iter: int = 100, rng=None) -> tuple:
    """
    Find the dominant eigenvector and eigenvalue via power iteration.

    Parameters
    ----------
    A : square symmetric matrix

    Returns
    -------
    (eigenvalue, eigenvector)
    """
    if rng is None:
        rng = np.random.default_rng()
    n = A.shape[0]
    v = rng.normal(0, 1, n)
    v = v / np.linalg.norm(v)
    for _ in range(n_iter):
        w = A @ v
        eigenvalue = v @ w     # Rayleigh quotient: vᵀAv / vᵀv (with ‖v‖=1)
        v = w / np.linalg.norm(w)
    return eigenvalue, v

# Test on a known 2×2 matrix
A_test = np.array([[3.0, 1.0], [1.0, 2.0]])  # symmetric
lam_pi, v_pi = power_iteration(A_test, rng=np.random.default_rng(42))
lam_np, v_np = np.linalg.eig(A_test)
idx = np.argmax(np.abs(lam_np))
print(f"Power iteration:  λ={lam_pi:.6f}, v={v_pi}")
print(f"numpy.linalg.eig: λ={lam_np[idx]:.6f}, v={v_np[:,idx]}")
print(f"\nVerify Av = λv: {np.allclose(A_test @ v_pi, lam_pi * v_pi, atol=1e-6)}")

In [ ]:
# Cell 6.2 — Eigendecomposition of a gene expression covariance matrix
rng = np.random.default_rng(42)

# Synthetic expression: 3 cell types with distinct profiles
n_samples, n_genes = 60, 500
cell_type = np.repeat([0, 1, 2], n_samples // 3)
type_profiles = rng.normal(0, 3, (3, n_genes))
X = type_profiles[cell_type] + rng.normal(0, 0.8, (n_samples, n_genes))

# Center and compute sample covariance (samples × samples)
X_centered = X - X.mean(axis=0)       # center genes
C = (X_centered @ X_centered.T) / (n_genes - 1)  # (samples × samples)

# Eigendecomposition (eigh for symmetric = more stable than eig)
eigenvalues, eigenvectors = np.linalg.eigh(C)
# eigh returns in ascending order — reverse for descending
idx_sorted = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx_sorted]
eigenvectors = eigenvectors[:, idx_sorted]

# Explained variance ratio
explained_var = eigenvalues / eigenvalues.sum()

print(f"Top 5 eigenvalues: {eigenvalues[:5].round(3)}")
print(f"Top 5 explained variance ratios: {explained_var[:5].round(4)}")
print(f"Cumulative variance (top 3): {explained_var[:3].sum():.2%}")
print(f"\nAll eigenvalues ≥ 0: {(eigenvalues >= -1e-10).all()}  [covariance matrix is PSD]")

In [ ]:
# Cell 6.3 — Project data onto first two eigenvectors (this IS PCA)
pc_scores = X_centered @ eigenvectors[:, :2]  # (samples, 2)
print(f"PC scores shape: {pc_scores.shape}  (samples × 2 PCs)")
print(f"PC1 explains {explained_var[0]:.1%}, PC2 explains {explained_var[1]:.1%} of variance")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Eigenvector geometry + PCA scatter + scree plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: 2×2 matrix transformation — show eigenvectors don't rotate
ax = axes[0]
theta = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])
ellipse = A_test @ circle   # transform the unit circle
ax.plot(*circle, 'steelblue', lw=1.5, label='Unit circle')
ax.plot(*ellipse, 'tomato', lw=1.5, label='Transformed')
lam_all, v_all = np.linalg.eig(A_test)
for lam, v in zip(lam_all, v_all.T):
    ax.annotate('', xy=lam*v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2))
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
ax.set_aspect('equal'); ax.set_xlim(-4,4); ax.set_ylim(-4,4)
ax.axhline(0, lw=0.5, color='gray'); ax.axvline(0, lw=0.5, color='gray')
ax.set_title('Eigenvectors only stretch, not rotate'); ax.legend(fontsize=8)

# Panel 2: PCA scatter PC1 vs PC2
ax = axes[1]
colors = ['steelblue', 'tomato', 'green']
for ct in range(3):
    mask = cell_type == ct
    ax.scatter(pc_scores[mask, 0], pc_scores[mask, 1], s=20, alpha=0.8,
               color=colors[ct], label=f'Cell type {ct}')
ax.set_xlabel(f'PC1 ({explained_var[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({explained_var[1]:.1%} variance)')
ax.set_title('PCA: 3 cell types separate in PC space')
ax.legend(fontsize=8)

# Panel 3: Scree plot
ax = axes[2]
k_show = 20
ax.bar(range(1, k_show+1), explained_var[:k_show], color='steelblue', alpha=0.8)
ax.plot(range(1, k_show+1), np.cumsum(explained_var[:k_show]),
        'tomato', lw=2, marker='o', ms=4, label='Cumulative')
ax.axhline(0.9, color='gray', ls='--', lw=1, label='90% threshold')
ax.set_xlabel('Principal component'); ax.set_ylabel('Variance explained')
ax.set_title('Scree plot — elbow at PC3')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `power_iteration(A)` from scratch. How many iterations does it take
   to converge on the test matrix? What slows convergence (hint: ratio of eigenvalues)?
2. The explained variance ratio for PC $i$ is $\lambda_i / \sum_j \lambda_j$.
   Derive why $\sum_i \lambda_i = \text{trace}(C)$ for a symmetric matrix $C$.
3. Change the number of cell types from 3 to 5. How many PCs are needed to
   explain 90% of variance? Is there a relationship between cell types and PCs?
4. What happens to the PCA result if you do NOT center the data before computing
   the covariance matrix? Demonstrate numerically.

---
## Quiz — Active Recall

1. Write the eigenvalue equation $A\mathbf{v} = \lambda \mathbf{v}$. What is special about an eigenvector?
2. For a symmetric positive semi-definite matrix (like a covariance matrix): what are
   the constraints on eigenvalues? Are eigenvectors orthogonal?
3. What is the explained variance ratio for PC $i$? Write the formula.
4. Why is `np.linalg.eigh` preferred over `np.linalg.eig` for covariance matrices?
5. What is the scree plot? How do you use it to choose the number of PCs?

---
## Reflection

**Date completed:** ____________________

> *[Could you explain to a biologist why eigenvectors of the covariance matrix are the directions of maximum variance? Where does the word 'principal component' come from?]*

---
*Next: `05_pca_from_first_principles.ipynb`*